# <b>ПЛАН Б: дообучение Айки в Google Colab</b>
Qwen2.5-1.5B-Instruct + QLoRA на 40к ваших диалогах.
GPU: T4 (меню «Среда выполнения» → «Сменить среду» → T4 GPU).
Всё сохраняется на Google Диск.

In [ ]:
# СРЕДА → СМЕНЫ СРЕДЫ → GPU T4, потом запускай всё по порядку
!nvidia-smi

In [ ]:
# 2. Ставим зависимости (~3 мин)
!pip install -q transformers==4.46.3 peft==0.13.2 bitsandbytes==0.44.1 accelerate==1.1.1 sentencepiece

In [ ]:
# 3. Подключаем Google Диск (появится ссылка — разреши доступ)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Забираем ваши 40к диалогов из репы (та самая aika-finetune)
!wget -q https://github.com/lucifermornngstar52-cell/aika-finetune/raw/main/data/train.jsonl -O train.jsonl
!wget -q https://github.com/lucifermornngstar52-cell/aika-finetune/raw/main/data/val.jsonl -O val.jsonl
import json
lines = open('train.jsonl', encoding='utf-8').read().strip().split('\n')
data = [json.loads(l)['messages'] for l in lines]
print(f"диалогов: {len(data):,} | пример: {data[0][0]['content'][:60]}...")

In [ ]:
# 5. Загружаем мозг Qwen в 4 битах (~1 ГБ, минуты 3)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
BASE = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(BASE)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map={"": 0})
model.config.use_cache = False
print("влезло в VRAM:", torch.cuda.memory_allocated()/1e9, "ГБ")

In [ ]:
# 6. LoRA-слой — тот самый «1% души»
from peft import LoraConfig, get_peft_model
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
                  task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# 7. Готовим данные в формат чата Qwen
import torch
class DialogDataset(torch.utils.data.Dataset):
    def __init__(self, msgs, tok, max_len=512):
        self.items = []
        for m in msgs:
            ids = tok.apply_chat_template(m, add_generation_prompt=False, return_tensors="pt")
            self.items.append(ids[0][:max_len])
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return {"input_ids": self.items[i]}

train_ds = DialogDataset(data, tok)
print(f"готово: {len(train_ds):,} примеров")

In [ ]:
# 8. ОБУЧЕНИЕ (~30-40 мин на T4). Уйти пить чай.
from transformers import TrainingArguments, Trainer
def collate(features):
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    maxlen = max(len(f["input_ids"]) for f in features)
    ids, labels, att = [], [], []
    for f in features:
        x = f["input_ids"].tolist()
        n = maxlen - len(x)
        ids.append(x + [pad]*n)
        labels.append(x + [-100]*n)
        att.append([1]*len(x) + [0]*n)
    return {"input_ids": torch.tensor(ids), "labels": torch.tensor(labels), "attention_mask": torch.tensor(att)}

args = TrainingArguments(
    output_dir="out", num_train_epochs=1, per_device_train_batch_size=4,
    gradient_accumulation_steps=8, learning_rate=2e-4, warmup_ratio=0.03,
    logging_steps=25, save_strategy="no", fp16=True, optim="paged_adamw_8bit",
    report_to="none", lr_scheduler_type="cosine", seed=42)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collate)
trainer.train()
print("ГОТОВО. Последний loss:", trainer.state.log_history[-1].get("train_loss", "?"))

In [ ]:
# 9. Проверяем, что она заговорила как надо
model.config.use_cache = True
model.eval()
def ask(q):
    ids = tok.apply_chat_template([{"role":"user","content":q}], add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=120, temperature=0.7, top_p=0.9, do_sample=True,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
for q in ["привет, как дела?", "что делать если скучно?", "расскажи о себе"]:
    print(f"ТЫ: {q}\nАЙКА: {ask(q)}\n")

In [ ]:
# 10. Сливаем LoRA в один мозг и сохраняем на Диск (~2 ГБ)
merged = model.merge_and_unload()
merged = merged.half()
save = "/content/drive/MyDrive/aika-qwen-1.5b"
merged.save_pretrained(save)
tok.save_pretrained(save)
print("сохранено:", save)
# файлы увидишь в Google Drive → папка aika-qwen-1.5b

In [ ]:
# 11. (опционально) GGUF для телефона / llama.cpp
# Компиляция ~7 мин, потом минуты 3.
!git clone -q https://github.com/ggerganov/llama.cpp /content/llamac
!pip install -q gguf
!cd /content/llamac && make -j2 llama-quant 2>&1 | tail -1
!python /content/llamac/convert_hf_to_gguf.py /content/drive/MyDrive/aika-qwen-1.5b --outfile /content/aika-f16.gguf --outtype f16
!/content/llamac/llama-quant /content/aika-f16.gguf /content/drive/MyDrive/aika-q4_k_m.gguf Q4_K_M
!ls -lh /content/drive/MyDrive/aika-q4_k_m.gguf